In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [5]:
load_dotenv(override=True)

# openai_api_key = os.getenv('OPEN')
openai_api_key = os.getenv('OPENROUTER_API_KEY')
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# MODEL = "mistralai/mistral-7b-instruct"
# MODEL = "openchat/openchat-3.5-0106"
MODEL = "openrouter/auto"
openai = OpenAI( api_key= openai_api_key, base_url= OPENROUTER_BASE_URL)

OpenAI API Key exists and begins sk-or-v1


In [2]:
DB = "itemsWithPrices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            item_name TEXT PRIMARY KEY,
            price REAL
        )
    ''')
    conn.commit()

# Insert products (ONE connection only)
products = [
    ("pintola peanut butter", 350),
    ("bambino pasta", 120),
    ("oats", 180),
    ("maggi", 15)
]

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executemany(
        'INSERT OR REPLACE INTO prices (item_name, price) VALUES (?, ?)',
        [(name.lower(), price) for name, price in products]
    )
    conn.commit()


In [3]:
def get_item_price(item_name):
    print(f"DATABASE TOOL CALLED: Getting price for {item_name}", flush=True)

    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT price FROM prices WHERE item_name = ?', 
            (item_name.lower(),)
        )
        result = cursor.fetchone()

        if result:
            return f"Price of {item_name} is ₹{result[0]}"
        else:
            return "No price data available for this item"

In [4]:
get_item_price("pintola peanut butter")

DATABASE TOOL CALLED: Getting price for pintola peanut butter


'Price of pintola peanut butter is ₹350.0'

In [6]:
system_message = """
You are an FMCG shop assistant.
Help users with product prices.
Keep answers short and clear.
"""

In [7]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_item_price",
            "description": "Get price of FMCG item",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_name": {"type": "string"}
                },
                "required": ["item_name"]
            }
        }
    }
]

In [ ]:
def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, max_tokens=300, temperature=0.7) 
    return response.choices[0].message.content
print(MODEL)
gr.ChatInterface(fn=chat, type="messages").launch()

In [8]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [10]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_item_price":
            arguments = json.loads(tool_call.function.arguments)
            item_name = arguments.get('item_name')
            price_details = get_item_price(item_name)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [11]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for oats


In [12]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image

In [13]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

ModuleNotFoundError: No module named 'gtts'

In [ ]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    # voice = talker(reply)

    if cities:
        image = artist(cities[0])
    
    return history, voice, image


In [16]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_item_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('item_name')
            cities.append(city)
            price_details = get_item_price(item_name)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

In [21]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        # image_output = gr.Image(height=500, interactive=False)
    # with gr.Row():
    #     audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "c:\Users\Pranav Jain\Desktop\Personal\AI_Course\projects\llm_engineering\.venv\Lib\site-packages\gradio\queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Pranav Jain\Desktop\Personal\AI_Course\projects\llm_engineering\.venv\Lib\site-packages\gradio\route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Pranav Jain\Desktop\Personal\AI_Course\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Pranav Jain\Desktop\Personal\AI_Course\projects\llm_engineering\.venv\Lib\site-packages\gradio\blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
       